# E1 · Umbrales de decisión por etiqueta sobre el stacking
**Parte de:** E0 (stacking de 5 modelos).

**Hipótesis.** El F1 de la clase positiva premia encontrar más positivos (*recall*). El stacking predice HS en el 39 % del test, menos que la prevalencia del train (44 %). Bajar el umbral de 0,5 podría subir el F1. En el notebook original los umbrales solo se probaron sobre el ensamble, no sobre el stacking.

**Cómo se mide sin hacer trampa.** Elegir el umbral con las mismas predicciones con las que se evalúa infla el resultado. Por eso se usa **CV anidada**: en cada partición, el umbral se elige con las otras 4. También se prueban umbrales fijos iguales para las tres etiquetas.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **Umbral** | Probabilidad a partir de la cual se predice 1 (por defecto 0,5) |
| **CV anidada** | Elegir el umbral en 4 particiones y evaluarlo en la 5.ª, rotando |
| **Recall / precision** | De los positivos reales, cuántos encuentro / de lo que predigo positivo, cuánto acierto |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
ST = {s: tb.stacking(d, tb.BASES_ORIGINALES, s, con_test=False)[0] for s in tb.SEEDS}
R = {'E0 · stacking, umbral 0,5': tb.evaluar(d, ST),
     'Umbral por etiqueta (CV anidada, rejilla 0,20-0,70)': tb.evaluar_anidado(d, ST),
     'Umbral por etiqueta (CV anidada, rejilla 0,30-0,60)': tb.evaluar_anidado(d, ST, np.round(np.arange(.30, .601, .02), 2))}
for t in [0.40, 0.45, 0.55]:
    R[f'Umbral fijo {t}'] = tb.evaluar(d, ST, (t, t, t))
tb.tabla(R)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
"E0 · stacking, umbral 0,5",0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813
"Umbral por etiqueta (CV anidada, rejilla 0,20-0,70)",0.8049,0.8316,0.7978,0.8114,0.0022,0.7633,0.814 / 0.812 / 0.808
"Umbral por etiqueta (CV anidada, rejilla 0,30-0,60)",0.8057,0.8315,0.7978,0.8117,0.0019,0.7643,0.814 / 0.812 / 0.809
Umbral fijo 0.4,0.8043,0.8323,0.7993,0.8120,0.0021,0.7691,0.810 / 0.815 / 0.811
Umbral fijo 0.45,0.8044,0.8334,0.7974,0.8117,0.0017,0.7718,0.814 / 0.812 / 0.810
Umbral fijo 0.55,0.8036,0.8331,0.7952,0.8106,0.0011,0.7784,0.811 / 0.809 / 0.812


### ¿Qué tan estables son los umbrales óptimos?

In [3]:
pd.DataFrame({s: tb.umbrales_optimos(d.Y, P, np.arange(d.n)) for s, P in ST.items()},
             index=tb.LABELS).T.rename_axis('semilla')

,HS,TR,AG
semilla,,,
42,0.48,0.54,0.52
43,0.44,0.56,0.40
44,0.34,0.62,0.38


## Conclusión: **no se adopta**
- Con CV anidada, los umbrales por etiqueta dan un F1 medio igual o menor que 0,5: suben HS un poco y bajan TR.
- Los umbrales óptimos cambian mucho de una semilla a otra (HS entre 0,34 y 0,48; AG entre 0,38 y 0,52). Es una señal de que se estaría ajustando ruido.
- El `class_weight='balanced'` del metamodelo ya mueve el punto de corte hacia la clase positiva, y por eso 0,5 queda cerca del óptimo.
- Se mantiene el umbral 0,5. No se genera submission.